# Motex V3（模型本体 · notebook 内装配）

自清洁的 Decoder-only 语言模型。与 v1/v2/v2_1/v2_2 一样，**模型的定义在 notebook 内完成**，
`motex_utils` 只提供共享构建块（基础算子 / 注意力组件 / MoE / 训练工具）。

## V3 相对前代的改进
1. **内置因果掩码**：训练/预填充/增量生成结构上自洽，不依赖外部 `valid_lens` 的传法（前代在
   `valid_lens=None/整句长度` 时训练会变双向——偷看未来、loss 虚低、生成必崩）。
2. 标准组合：Pre-Norm(RMSNorm) + GQA(RoPE+KV-Cache) + SwiGLU FFN + 权重绑定。
3. 统一返回 `(logits, state, aux_loss)`，兼容 `motex_utils.training` 的训练/推理接口。

In [ ]:
# 仓库根目录加入 sys.path（本仓库自包含）
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

import torch
from torch import nn

# 共享构建块（motex_utils 只放这些）
from motex_utils.transformer import RMSNorm
from motex_utils.attention import GQARopeCausalAttention
from motex_utils.moe import SwiGLUMLP

In [ ]:
# ============ 模型装配（notebook 内定义，与 v1/v2 风格一致）============
class MotexV3Block(nn.Module):
    def __init__(self, d_model, num_heads, num_kv_heads, ffn_hidden, dropout, i, max_seq_len,
                 rope_scaling=None, use_sdp=False, qk_norm=True):
        super().__init__()
        self.i = i
        self.norm1 = RMSNorm(d_model)
        # 内置因果掩码 + 可选改进（参数说明见 motex_utils.attention.GQARopeCausalAttention）
        self.attn = GQARopeCausalAttention(d_model, num_heads, num_kv_heads, dropout, max_seq_len,
                                          rope_scaling=rope_scaling, use_sdp=use_sdp, qk_norm=qk_norm)
        self.norm2 = RMSNorm(d_model)
        self.ffn = SwiGLUMLP(d_model, ffn_hidden, dropout)

    def forward(self, x, state=None, valid_lens=None):
        h = self.norm1(x)
        attn_out, state = self.attn(h, state, self.i)
        x = x + attn_out
        x = x + self.ffn(self.norm2(x))
        return x, state, 0.0   # 非 MoE，aux_loss=0


class MotexV3Decoder(nn.Module):
    def __init__(self, vocab_size, d_model, num_layers, num_heads, num_kv_heads,
                 ffn_hidden, dropout, max_seq_len, rope_scaling=None, use_sdp=False, qk_norm=True):
        super().__init__()
        self.num_layers = num_layers
        self.d_model = d_model
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.blks = nn.ModuleList([
            MotexV3Block(d_model, num_heads, num_kv_heads, ffn_hidden, dropout, i, max_seq_len,
                         rope_scaling=rope_scaling, use_sdp=use_sdp, qk_norm=qk_norm)
            for i in range(num_layers)
        ])

    def forward(self, tokens, valid_lens=None, state=None):
        x = self.token_emb(tokens)
        if state is not None and state[0] is None:
            state[0] = [None] * self.num_layers
        aux = 0.0
        for blk in self.blks:
            x, state, a = blk(x, state, valid_lens)
            aux += a
        return x, state, aux / len(self.blks)


class MotexV3(nn.Module):
    """统一接口：forward(tokens, valid_lens=None, state=None) -> (logits, state, aux_loss)"""
    def __init__(self, vocab_size, d_model=512, num_layers=8, num_heads=8, num_kv_heads=2,
                 ffn_hidden=1024, dropout=0.1, max_seq_len=256,
                 rope_scaling=None, use_sdp=False, qk_norm=True):
        super().__init__()
        self.vocab_size = vocab_size
        self.d_model = d_model
        self.decoder = MotexV3Decoder(vocab_size, d_model, num_layers, num_heads, num_kv_heads,
                                     ffn_hidden, dropout, max_seq_len,
                                     rope_scaling=rope_scaling, use_sdp=use_sdp, qk_norm=qk_norm)
        self.norm = RMSNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.decoder.token_emb.weight   # 权重绑定
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Embedding):
                nn.init.normal_(m.weight, 0.0, 0.02)

    def forward(self, tokens, valid_lens=None, state=None):
        x, state, aux = self.decoder(tokens, valid_lens, state)
        return self.lm_head(self.norm(x)), state, aux

In [ ]:
# ============ 生成（KV-Cache 增量解码，notebook 内定义）============
from torch.nn import functional as F

@torch.no_grad()
def generate(net, tokenizer, prompt, max_new_tokens, device, temperature=0.8, top_k=40,
             repetition_penalty=1.15):
    """tokenizer 需提供 encode(text)->list[int] 与 stoi/itos"""
    net.eval()
    ids = tokenizer.encode(prompt)
    special = {tokenizer.stoi[s] for s in ('<pad>', '<bos>', '<eos>') if s in tokenizer.stoi}
    state = [[None] * net.decoder.num_layers, [None] * net.decoder.num_layers]
    logits, state, _ = net(torch.tensor([ids], device=device), None, state)

    def pick(logts, seen):
        logts = logts.clone()
        if repetition_penalty != 1.0 and seen:
            for t in seen:
                logts[t] /= repetition_penalty
        p = F.softmax(logts / temperature, dim=-1)
        top = torch.topk(p, min(top_k, p.numel()))
        return top.indices[torch.multinomial(top.values, 1)].item()

    seen = []; gen = []
    nxt = pick(logits[0, -1, :], seen)
    for _ in range(max_new_tokens):
        gen.append(nxt); seen.append(nxt)
        logits, state, _ = net(torch.tensor([[nxt]], device=device), None, state)
        nxt = pick(logits[0, -1, :], seen)
    return prompt + ''.join(tokenizer.itos[i] for i in gen if i not in special), gen

In [ ]:
# ============ 使用演示 ============
net = MotexV3(vocab_size=4096, d_model=256, num_layers=4, num_heads=4,
              num_kv_heads=2, ffn_hidden=512, dropout=0.1, max_seq_len=128)
print('参数(M):', round(sum(p.numel() for p in net.parameters()) / 1e6, 2))
print('注意力:', type(net.decoder.blks[0].attn).__name__, '| qk_norm =', net.decoder.blks[0].attn.qk_norm)

x = torch.randint(5, 4096, (2, 128))
logits, state, aux = net(x, None, None)
print('train 前向 logits', tuple(logits.shape), ' 返回 3 元组，aux=', float(aux))

In [ ]:
# 生成演示（内置迷你字符分词器，仅演示接口；真实分词/训练在 dev/）
class MiniTok:
    def __init__(self, chars):
        self.stoi = {s: i for i, s in enumerate(['<pad>', '<unk>', '<bos>', '<eos>', '\n'] + chars)}
        self.itos = {v: k for k, v in self.stoi.items()}
    def encode(self, t): return [self.stoi.get(c, 1) for c in t]

tok = MiniTok(list('你我他她说目的地方向山里人大'))
net2 = MotexV3(vocab_size=len(tok.stoi), d_model=128, num_layers=2, num_heads=4,
               num_kv_heads=1, ffn_hidden=256, dropout=0.0, max_seq_len=64).eval()
print(generate(net2, tok, '你', 30, 'cpu', temperature=0.9, top_k=10, repetition_penalty=1.1)[0][:40])